## 5.0 COMPLETE MATHEMATICAL FORMULATION

This section extends the simplified model to every destination area, bicycle category, and feasible marginal-profit position in the workbook.

### Scope and explicit hypotheses

1. There is one source area and one relocation operation with a single global truck-capacity limit.
2. A bicycle may remain at the source; neither the truck nor any destination has a minimum-fill requirement.
3. Expected profits are deterministic, non-negative marginal values in the workbook's original units. They are not sorted, normalized, or assumed to be monotone, concave, or convex.
4. For each area-category pair, selecting position $n$ requires selecting every position $1,\ldots,n-1$.
5. A missing trailing marginal-profit value is interpreted as zero, as required by the case. Internal missing values are rejected because they would make the sequence ambiguous.
6. No destination capacity, routing cost, balancing target, or fairness rule is added because none is supplied by the case.

## 5.1 PREPARE INPUT

The complete model discovers areas from sheet-name suffixes and categories from the Categories sheet. Validation preserves the row position of every marginal profit, including a genuine zero value, and confirms that missing values occur only as trailing padding.

In [28]:
# 5.1 DATA-DRIVEN INPUT CONSOLIDATION AND VALIDATION

import math
import re

FULL_TRUCK_CAPACITY = 80.0
AREA_SHEET_PATTERN = re.compile(r"ExpectedProfitsArea(\d+)")

if "Categories" not in sheets:
    raise ValueError("The workbook must contain a 'Categories' sheet.")

full_categories_df = sheets["Categories"].copy()

if full_categories_df.shape[0] != 2:
    raise ValueError("The 'Categories' sheet must have exactly two data rows.")

full_categories = list(full_categories_df.columns)

if len(full_categories) != len(set(full_categories)):
    raise ValueError("Bicycle category names must be unique.")

surplus_values = pd.to_numeric(full_categories_df.iloc[0], errors="raise")
space_values = pd.to_numeric(full_categories_df.iloc[1], errors="raise")

if surplus_values.isna().any() or not all(float(value).is_integer() for value in surplus_values):
    raise ValueError("Every category surplus must be a non-negative integer.")

if (surplus_values < 0).any():
    raise ValueError("Category surplus cannot be negative.")

if space_values.isna().any() or (space_values <= 0).any():
    raise ValueError("Truck-space coefficients must be strictly positive.")

full_surplus = {category: int(surplus_values[category]) for category in full_categories}
full_space = {category: float(space_values[category]) for category in full_categories}

full_profit_frames = {}

for sheet_name, frame in sheets.items():
    if not sheet_name.startswith("ExpectedProfitsArea"):
        continue

    match = AREA_SHEET_PATTERN.fullmatch(sheet_name)

    if match is None:
        raise ValueError(f"Invalid expected-profit sheet name: {sheet_name}")

    area = int(match.group(1))

    if area in full_profit_frames:
        raise ValueError(f"Duplicate destination area identifier: {area}")

    full_profit_frames[area] = frame.copy()

if not full_profit_frames:
    raise ValueError("No expected-profit area sheets were found.")

full_areas = sorted(full_profit_frames)
full_observed_length = {}

for area in full_areas:
    frame = full_profit_frames[area]

    if list(frame.columns) != full_categories:
        raise ValueError(f"Area {area} category columns do not match the Categories sheet.")

    if not frame.index.equals(pd.RangeIndex(len(frame))):
        raise ValueError(f"Area {area} must use consecutive row positions.")

    for category in full_categories:
        series = frame[category]
        valid_positions = [
            position
            for position, value in enumerate(series, start=1)
            if pd.notna(value)
        ]

        observed_length = max(valid_positions, default=0)

        if valid_positions != list(range(1, observed_length + 1)):
            raise ValueError(
                f"Internal missing profit found for Area {area}, {category}."
            )

        observed_values = pd.to_numeric(
            series.iloc[:observed_length],
            errors="raise"
        ).astype(float)

        if not observed_values.map(math.isfinite).all():
            raise ValueError(f"Non-finite profit found for Area {area}, {category}.")

        if (observed_values < 0).any():
            raise ValueError(f"Negative profit found for Area {area}, {category}.")

        full_observed_length[area, category] = observed_length

# Exact capacity presolve: a later position cannot fit if all its predecessors do not fit.
full_horizon = {
    category: min(
        full_surplus[category],
        math.floor(FULL_TRUCK_CAPACITY / full_space[category] + 1e-9)
    )
    for category in full_categories
}

full_index = [
    (area, category, position)
    for area in full_areas
    for category in full_categories
    for position in range(1, full_horizon[category] + 1)
]

full_profit = {}

for area, category, position in full_index:
    if position <= full_observed_length[area, category]:
        value = full_profit_frames[area].iloc[position - 1][category]
        full_profit[area, category, position] = float(value)
    else:
        # Missing trailing values have zero marginal profit.
        full_profit[area, category, position] = 0.0


In [29]:
full_category_inputs = pd.DataFrame({
    "Surplus": pd.Series(full_surplus),
    "Truck Space per Bicycle": pd.Series(full_space),
    "Largest Capacity-Feasible Position": pd.Series(full_horizon)
})

full_length_table = pd.DataFrame(
    {
        area: {
            category: full_observed_length[area, category]
            for category in full_categories
        }
        for area in full_areas
    }
).T
full_length_table.index.name = "Area"

full_area_scale = pd.DataFrame([
    {
        "Area": area,
        "Observed Marginal Values": int(full_profit_frames[area].count().sum()),
        "Mean Marginal Profit": full_profit_frames[area].stack().mean(),
        "Maximum Marginal Profit": full_profit_frames[area].stack().max()
    }
    for area in full_areas
]).set_index("Area")

print(
    f"Validated {len(full_areas)} areas, {len(full_categories)} categories, "
    f"and {sum(full_observed_length.values()):,} observed marginal profits."
)
display(full_category_inputs)
display(full_length_table)
display(full_area_scale.round(4))

Validated 7 areas, 6 categories, and 3,321 observed marginal profits.


,Surplus,Truck Space per Bicycle,Largest Capacity-Feasible Position
Child,272,1.0,80
Adult,270,1.5,53
Electric,279,1.5,53
Racing,267,1.7,47
Mountain,282,1.7,47
Tricycle,279,4.0,20


,Child,Adult,Electric,Racing,Mountain,Tricycle
Area,,,,,,
1,114,115,102,109,108,106
2,53,59,71,55,63,68
3,96,93,111,93,107,106
4,105,97,78,105,108,109
5,49,41,44,46,42,48
6,80,84,97,74,81,71
7,50,56,59,57,59,52


,Observed Marginal Values,Mean Marginal Profit,Maximum Marginal Profit
Area,,,
1,654,25.4423,52.9781
2,369,0.4931,0.9992
3,606,49.5532,96.9476
4,602,48.5941,96.8895
5,270,0.5119,0.9998
6,487,43.1635,86.8887
7,333,0.4973,0.9940


## 5.2 PARAMETERS

### Mathematical formulation

Sets:

$$A = \{\text{destination areas}\}, \qquad C = \{\text{bicycle categories}\}$$

$$K_c = \{1,\ldots,S_c\} \qquad \forall c \in C$$

Parameters:

$$S_c = \text{surplus available for category }c$$

$$W_c = \text{truck space occupied by one bicycle of category }c$$

$$P_{a,c,n} = \text{marginal expected profit of position }n$$

$$T = 80 = \text{total truck capacity}$$

When the workbook has no value for a feasible position, $P_{a,c,n}=0$. In the implementation, positions are capped at

$$\bar K_c = \min\left(S_c,\left\lfloor\frac{T}{W_c}\right\rfloor\right).$$

This is an exact presolve reduction: selecting a position beyond $\bar K_c$ would require enough predecessor bicycles of the same category to exceed the truck capacity.

In [30]:
# 5.2 COMPLETE SETS AND PARAMETERS

full_model = pyo.ConcreteModel(name="RentalBike_Complete_Relocation")

full_model.A = pyo.Set(initialize=full_areas, ordered=True)
full_model.C = pyo.Set(initialize=full_categories, ordered=True)
full_model.I = pyo.Set(dimen=3, initialize=full_index, ordered=True)
full_model.I_SEQUENCE = pyo.Set(
    dimen=3,
    initialize=[
        (area, category, position)
        for area, category, position in full_index
        if position >= 2
    ],
    ordered=True
)

full_model.S = pyo.Param(
    full_model.C,
    initialize=full_surplus,
    within=pyo.NonNegativeIntegers
)

full_model.W = pyo.Param(
    full_model.C,
    initialize=full_space,
    within=pyo.PositiveReals
)

full_model.P = pyo.Param(
    full_model.I,
    initialize=full_profit,
    within=pyo.NonNegativeReals
)

full_model.T = pyo.Param(
    initialize=FULL_TRUCK_CAPACITY,
    within=pyo.PositiveReals
)

print(f"Binary index positions: {len(full_model.I):,}")
print(f"Sequential links: {len(full_model.I_SEQUENCE):,}")

Binary index positions: 2,100
Sequential links: 2,058


## 5.3 DECISION VARIABLES

### Mathematical formulation

$$y_{a,c,n} \in \{0,1\}$$

$y_{a,c,n}=1$ if marginal position $n$ of category $c$ is relocated to destination area $a$, and zero otherwise. The reported relocation quantity is derived as

$$X_{a,c}=\sum_{n \in K_c} y_{a,c,n}.$$

In [31]:
# 5.3 COMPLETE DECISION VARIABLES

full_model.y = pyo.Var(
    full_model.I,
    domain=pyo.Binary
)

## 5.4 CONSTRAINTS

### 5.4.1 Bicycle availability

$$\sum_{a \in A}\sum_{n \in K_c}y_{a,c,n}\leq S_c \qquad \forall c\in C$$

The surplus of each category is shared across all destinations.

### 5.4.2 Truck capacity

$$\sum_{a \in A}\sum_{c \in C}\sum_{n \in K_c}W_cy_{a,c,n}\leq T$$

There is one global truck-capacity constraint.

### 5.4.3 Sequential relocation

$$y_{a,c,n}\leq y_{a,c,n-1} \qquad \forall a\in A,\;c\in C,\;n=2,\ldots,|K_c|$$

This forces every selected sequence to be a prefix, without assuming any shape for the marginal-profit series.

In [32]:
# 5.4 COMPLETE CONSTRAINTS

def full_availability_rule(model, category):
    return sum(
        model.y[area, indexed_category, position]
        for area, indexed_category, position in model.I
        if indexed_category == category
    ) <= model.S[category]


full_model.availability_constraint = pyo.Constraint(
    full_model.C,
    rule=full_availability_rule
)


full_model.capacity_constraint = pyo.Constraint(
    expr=sum(
        full_model.W[category] * full_model.y[area, category, position]
        for area, category, position in full_model.I
    ) <= full_model.T
)


def full_sequence_rule(model, area, category, position):
    return (
        model.y[area, category, position]
        <= model.y[area, category, position - 1]
    )


full_model.sequence_constraint = pyo.Constraint(
    full_model.I_SEQUENCE,
    rule=full_sequence_rule
)

## 5.5 OBJECTIVE FUNCTION

### Mathematical formulation

$$\max Z=\sum_{a \in A}\sum_{c \in C}\sum_{n \in K_c}P_{a,c,n}y_{a,c,n}$$

The objective adds the original marginal values for every selected prefix position and maximizes total expected profit.

In [33]:
# 5.5 COMPLETE OBJECTIVE FUNCTION

full_model.objective = pyo.Objective(
    expr=sum(
        full_model.P[area, category, position]
        * full_model.y[area, category, position]
        for area, category, position in full_model.I
    ),
    sense=pyo.maximize
)

## 5.6 OUTPUT AND VALIDATION

HiGHS solves the binary MILP with a zero relative optimality gap. The results below report the operational relocation plan, objective value, capacity use, category availability, and destination contributions. Post-solve checks independently recompute the objective and verify integrality, prefix order, availability, and capacity.

In [34]:
# 5.6 SOLVE THE COMPLETE MODEL

full_solver = pyo.SolverFactory("appsi_highs")

if not full_solver.available(exception_flag=False):
    raise RuntimeError(
        "HiGHS is unavailable. Install pyomo and highspy before running the model."
    )

full_solver.options["mip_rel_gap"] = 0.0

full_results = full_solver.solve(
    full_model,
    tee=False
)

full_termination = full_results.solver.termination_condition

if full_termination != pyo.TerminationCondition.optimal:
    raise RuntimeError(f"The complete model did not solve to optimality: {full_termination}")

print("Solver Status:", full_results.solver.status)
print("Termination Condition:", full_termination)

Solver Status: ok
Termination Condition: optimal


In [35]:
# TRANSLATE BINARY POSITIONS INTO AN OPERATIONAL PLAN

full_selected = {
    (area, category, position): int(
        pyo.value(full_model.y[area, category, position]) > 0.5
    )
    for area, category, position in full_model.I
}

full_plan_rows = []

for area in full_areas:
    for category in full_categories:
        positions = range(1, full_horizon[category] + 1)
        selected_positions = [
            position
            for position in positions
            if full_selected[area, category, position] == 1
        ]
        quantity = len(selected_positions)
        expected_profit = sum(
            full_profit[area, category, position]
            for position in selected_positions
        )

        full_plan_rows.append({
            "Area": area,
            "Category": category,
            "Bicycles Relocated": quantity,
            "Truck Space Used": quantity * full_space[category],
            "Expected Profit": expected_profit
        })

full_plan = pd.DataFrame(full_plan_rows)
full_positive_plan = full_plan.loc[
    full_plan["Bicycles Relocated"] > 0
].reset_index(drop=True)

full_objective_value = float(pyo.value(full_model.objective))
full_capacity_used = float(full_plan["Truck Space Used"].sum())
full_capacity_remaining = FULL_TRUCK_CAPACITY - full_capacity_used
full_bicycles_relocated = int(full_plan["Bicycles Relocated"].sum())

In [36]:
# BUSINESS-READY RESULTS

full_summary = pd.DataFrame({
    "Metric": [
        "Optimal expected profit",
        "Bicycles relocated",
        "Truck capacity used",
        "Truck capacity remaining"
    ],
    "Value": [
        full_objective_value,
        full_bicycles_relocated,
        full_capacity_used,
        full_capacity_remaining
    ]
})

full_quantity_matrix = full_plan.pivot(
    index="Area",
    columns="Category",
    values="Bicycles Relocated"
).reindex(index=full_areas, columns=full_categories).astype(int)

full_category_summary = pd.DataFrame([
    {
        "Category": category,
        "Available": full_surplus[category],
        "Relocated": int(
            full_plan.loc[
                full_plan["Category"] == category,
                "Bicycles Relocated"
            ].sum()
        ),
        "Remaining at Source": full_surplus[category] - int(
            full_plan.loc[
                full_plan["Category"] == category,
                "Bicycles Relocated"
            ].sum()
        )
    }
    for category in full_categories
]).set_index("Category")

full_area_summary = full_plan.groupby("Area", as_index=True).agg(
    Bicycles_Relocated=("Bicycles Relocated", "sum"),
    Truck_Space_Used=("Truck Space Used", "sum"),
    Expected_Profit=("Expected Profit", "sum")
)

display(full_summary.round(4))
display(
    full_positive_plan.style.format({
        "Truck Space Used": "{:.1f}",
        "Expected Profit": "{:,.4f}"
    })
)
display(full_quantity_matrix)
display(full_category_summary)
display(full_area_summary.round(4))

,Metric,Value
0,Optimal expected profit,4358.7834
1,Bicycles relocated,79.0000
2,Truck capacity used,80.0000
3,Truck capacity remaining,0.0000


,Area,Category,Bicycles Relocated,Truck Space Used,Expected Profit
0,1,Child,1,1.0,45.4865
1,3,Child,69,69.0,"3,769.8666"
2,3,Adult,1,1.5,70.1450
3,4,Child,7,7.0,392.0342
4,6,Adult,1,1.5,81.2511


Category,Child,Adult,Electric,Racing,Mountain,Tricycle
Area,,,,,,
1,1,0,0,0,0,0
2,0,0,0,0,0,0
3,69,1,0,0,0,0
4,7,0,0,0,0,0
5,0,0,0,0,0,0
6,0,1,0,0,0,0
7,0,0,0,0,0,0


,Available,Relocated,Remaining at Source
Category,,,
Child,272,77,195
Adult,270,2,268
Electric,279,0,279
Racing,267,0,267
Mountain,282,0,282
Tricycle,279,0,279


,Bicycles_Relocated,Truck_Space_Used,Expected_Profit
Area,,,
1,1,1.0,45.4865
2,0,0.0,0.0000
3,70,70.5,3840.0116
4,7,7.0,392.0342
5,0,0.0,0.0000
6,1,1.5,81.2511
7,0,0.0,0.0000


In [37]:
# POST-SOLVE VALIDATION

VALIDATION_TOLERANCE = 1e-6

full_recomputed_profit = sum(
    full_profit[area, category, position] * selected
    for (area, category, position), selected in full_selected.items()
)

full_integrality_ok = all(
    abs(pyo.value(full_model.y[index]) - round(pyo.value(full_model.y[index])))
    <= VALIDATION_TOLERANCE
    for index in full_model.I
)

full_prefix_ok = all(
    full_selected[area, category, position]
    <= full_selected[area, category, position - 1]
    for area, category, position in full_model.I_SEQUENCE
)

full_availability_ok = all(
    full_plan.loc[
        full_plan["Category"] == category,
        "Bicycles Relocated"
    ].sum() <= full_surplus[category]
    for category in full_categories
)

full_capacity_ok = (
    full_capacity_used <= FULL_TRUCK_CAPACITY + VALIDATION_TOLERANCE
)

full_missing_tail_selected = sum(
    selected
    for (area, category, position), selected in full_selected.items()
    if position > full_observed_length[area, category]
)

full_validation = pd.DataFrame([
    {
        "Check": "Solver proved optimality",
        "Passed": full_termination == pyo.TerminationCondition.optimal,
        "Detail": str(full_termination)
    },
    {
        "Check": "Binary integrality",
        "Passed": full_integrality_ok,
        "Detail": f"{len(full_model.I):,} variables checked"
    },
    {
        "Check": "Sequential-prefix constraints",
        "Passed": full_prefix_ok,
        "Detail": f"{len(full_model.I_SEQUENCE):,} links checked"
    },
    {
        "Check": "Category availability",
        "Passed": full_availability_ok,
        "Detail": "All category totals are within source surplus"
    },
    {
        "Check": "Truck capacity",
        "Passed": full_capacity_ok,
        "Detail": f"{full_capacity_used:.1f} used out of {FULL_TRUCK_CAPACITY:.1f}"
    },
    {
        "Check": "Objective recomputation",
        "Passed": abs(full_objective_value - full_recomputed_profit)
        <= VALIDATION_TOLERANCE,
        "Detail": f"Recomputed value = {full_recomputed_profit:.4f}"
    },
    {
        "Check": "Missing-profit tail selections",
        "Passed": full_missing_tail_selected == 0,
        "Detail": f"{full_missing_tail_selected} zero-profit tail positions selected"
    }
])

display(full_validation)

if not full_validation["Passed"].all():
    failed_checks = full_validation.loc[
        ~full_validation["Passed"],
        "Check"
    ].tolist()
    raise AssertionError(f"Post-solve validation failed: {failed_checks}")

print(f"Validated optimal objective: {full_objective_value:,.4f}")

,Check,Passed,Detail
0,Solver proved optimality,True,optimal
1,Binary integrality,True,"2,100 variables checked"
2,Sequential-prefix constraints,True,"2,058 links checked"
3,Category availability,True,All category totals are within source surplus
4,Truck capacity,True,80.0 used out of 80.0
5,Objective recomputation,True,Recomputed value = 4358.7834
6,Missing-profit tail selections,True,0 zero-profit tail positions selected


Validated optimal objective: 4,358.7834


## 6.0 BASELINE COMPARISON

The exact MILP is compared with two deterministic, explainable baselines. Both heuristics use the same workbook inputs, respect category availability and truck capacity, and construct prefix-feasible relocation quantities. They differ only in how they choose the next allocation.

- The average-profit-density heuristic ranks complete accessible area-category prefixes by their average expected profit per unit of truck space, then fills the truck in that fixed order.
- The next-marginal greedy heuristic repeatedly selects the currently accessible next bicycle with the highest marginal expected profit per unit of truck space.

These baselines are feasible decision rules, not alternative optimality claims. Their gaps quantify the additional value obtained from jointly considering non-monotonic future marginal profits.

In [38]:
# 6.0 COMMON HEURISTIC EVALUATION

# All truck-space values have one decimal place, so integer units avoid
# floating-point ambiguity in the constructive heuristics.
HEURISTIC_SPACE_SCALE = 10
heuristic_capacity_units = int(round(FULL_TRUCK_CAPACITY * HEURISTIC_SPACE_SCALE))
heuristic_space_units = {
    category: int(round(full_space[category] * HEURISTIC_SPACE_SCALE))
    for category in full_categories
}

if any(
    abs(
        heuristic_space_units[category] / HEURISTIC_SPACE_SCALE
        - full_space[category]
    ) > VALIDATION_TOLERANCE
    for category in full_categories
):
    raise ValueError("Truck-space scaling is not exact for the workbook values.")


def evaluate_prefix_quantities(method_name, quantities):
    """Validate area-category prefix quantities and calculate their results."""

    normalized_quantities = {}

    for area in full_areas:
        for category in full_categories:
            raw_quantity = quantities.get((area, category), 0)
            quantity = int(raw_quantity)

            if quantity != raw_quantity or quantity < 0:
                raise ValueError(
                    f"{method_name} produced an invalid quantity for "
                    f"Area {area}, {category}: {raw_quantity}"
                )

            if quantity > full_horizon[category]:
                raise ValueError(
                    f"{method_name} exceeded the feasible position horizon for "
                    f"Area {area}, {category}."
                )

            normalized_quantities[area, category] = quantity

    category_totals = {
        category: sum(
            normalized_quantities[area, category]
            for area in full_areas
        )
        for category in full_categories
    }

    if any(
        category_totals[category] > full_surplus[category]
        for category in full_categories
    ):
        raise ValueError(f"{method_name} violates category availability.")

    capacity_units_used = sum(
        normalized_quantities[area, category]
        * heuristic_space_units[category]
        for area in full_areas
        for category in full_categories
    )

    if capacity_units_used > heuristic_capacity_units:
        raise ValueError(f"{method_name} violates truck capacity.")

    plan_rows = []

    for area in full_areas:
        for category in full_categories:
            quantity = normalized_quantities[area, category]
            expected_profit = sum(
                full_profit[area, category, position]
                for position in range(1, quantity + 1)
            )

            plan_rows.append({
                "Area": area,
                "Category": category,
                "Bicycles Relocated": quantity,
                "Truck Space Used": quantity * full_space[category],
                "Expected Profit": expected_profit
            })

    plan = pd.DataFrame(plan_rows)
    positive_plan = plan.loc[
        plan["Bicycles Relocated"] > 0
    ].reset_index(drop=True)

    return {
        "Method": method_name,
        "Quantities": normalized_quantities,
        "Plan": plan,
        "Positive Plan": positive_plan,
        "Expected Profit": float(plan["Expected Profit"].sum()),
        "Bicycles Relocated": int(plan["Bicycles Relocated"].sum()),
        "Truck Capacity Used": capacity_units_used / HEURISTIC_SPACE_SCALE,
        "Truck Capacity Remaining": (
            heuristic_capacity_units - capacity_units_used
        ) / HEURISTIC_SPACE_SCALE,
        "Feasible": True
    }


### 6.1 Average-Profit-Density Heuristic

For each area-category pair, the heuristic calculates the mean of its capacity-accessible observed marginal-profit prefix and divides it by the category's truck-space coefficient. It ranks pairs once by this score and assigns as many prefix bicycles as possible to each pair in rank order. Missing trailing zero-profit positions are not deliberately loaded.

In [39]:
# 6.1 AVERAGE-PROFIT-DENSITY HEURISTIC

category_order = {
    category: position
    for position, category in enumerate(full_categories)
}

average_density_rows = []

for area in full_areas:
    for category in full_categories:
        accessible_positions = min(
            full_observed_length[area, category],
            full_horizon[category]
        )

        if accessible_positions == 0:
            average_profit = 0.0
        else:
            average_profit = sum(
                full_profit[area, category, position]
                for position in range(1, accessible_positions + 1)
            ) / accessible_positions

        average_density_rows.append({
            "Area": area,
            "Category": category,
            "Accessible Positions": accessible_positions,
            "Average Marginal Profit": average_profit,
            "Average Profit per Space": average_profit / full_space[category]
        })

average_density_ranking = pd.DataFrame(average_density_rows).sort_values(
    by=[
        "Average Profit per Space",
        "Average Marginal Profit",
        "Area"
    ],
    ascending=[False, False, True],
    kind="stable"
).reset_index(drop=True)

average_quantities = {
    (area, category): 0
    for area in full_areas
    for category in full_categories
}
average_category_used = {category: 0 for category in full_categories}
average_capacity_remaining = heuristic_capacity_units

for candidate in average_density_ranking.to_dict(orient="records"):
    if candidate["Average Profit per Space"] <= 0:
        break

    area = candidate["Area"]
    category = candidate["Category"]
    maximum_quantity = min(
        candidate["Accessible Positions"],
        full_surplus[category] - average_category_used[category],
        average_capacity_remaining // heuristic_space_units[category]
    )
    maximum_quantity = int(maximum_quantity)

    if maximum_quantity <= 0:
        continue

    average_quantities[area, category] += maximum_quantity
    average_category_used[category] += maximum_quantity
    average_capacity_remaining -= (
        maximum_quantity * heuristic_space_units[category]
    )

average_result = evaluate_prefix_quantities(
    "Average-profit-density heuristic",
    average_quantities
)

display(average_density_ranking.head(10).round(4))
display(average_result["Positive Plan"].round(4))

,Area,Category,Accessible Positions,Average Marginal Profit,Average Profit per Space
0,3,Child,80,52.9650,52.9650
1,4,Child,80,49.4962,49.4962
2,6,Child,80,43.8924,43.8924
3,3,Adult,53,49.3786,32.9191
4,4,Electric,53,48.6198,32.4132
5,4,Mountain,47,54.2918,31.9364
6,3,Electric,53,47.8956,31.9304
7,3,Racing,47,52.7232,31.0137
8,6,Adult,53,45.0156,30.0104
9,6,Electric,53,44.2847,29.5231


,Area,Category,Bicycles Relocated,Truck Space Used,Expected Profit
0,3,Child,80,80.0,4237.1973


### 6.2 Next-Marginal Greedy Heuristic

Starting from zero relocation, this heuristic exposes only the next position of every area-category sequence. At each iteration, it selects the feasible next bicycle with the largest marginal expected profit per truck-space unit. Because it never skips a position, its output always satisfies the prefix rule; however, it cannot look through a low current value to recognize valuable later positions.

In [40]:
# 6.2 NEXT-MARGINAL GREEDY HEURISTIC

greedy_quantities = {
    (area, category): 0
    for area in full_areas
    for category in full_categories
}
greedy_category_used = {category: 0 for category in full_categories}
greedy_capacity_remaining = heuristic_capacity_units
greedy_decision_log = []
greedy_iteration = 0

while True:
    candidates = []

    for area in full_areas:
        for category in full_categories:
            next_position = greedy_quantities[area, category] + 1
            last_observed_position = min(
                full_observed_length[area, category],
                full_horizon[category]
            )

            if next_position > last_observed_position:
                continue

            if greedy_category_used[category] >= full_surplus[category]:
                continue

            if heuristic_space_units[category] > greedy_capacity_remaining:
                continue

            marginal_profit = full_profit[area, category, next_position]

            candidates.append({
                "Area": area,
                "Category": category,
                "Position": next_position,
                "Marginal Profit": marginal_profit,
                "Profit per Space": marginal_profit / full_space[category]
            })

    if not candidates:
        break

    best_candidate = sorted(
        candidates,
        key=lambda candidate: (
            -candidate["Profit per Space"],
            -candidate["Marginal Profit"],
            heuristic_space_units[candidate["Category"]],
            candidate["Area"],
            category_order[candidate["Category"]]
        )
    )[0]

    area = best_candidate["Area"]
    category = best_candidate["Category"]
    greedy_iteration += 1
    greedy_quantities[area, category] += 1
    greedy_category_used[category] += 1
    greedy_capacity_remaining -= heuristic_space_units[category]
    greedy_decision_log.append({
        "Iteration": greedy_iteration,
        **best_candidate,
        "Capacity Remaining": (
            greedy_capacity_remaining / HEURISTIC_SPACE_SCALE
        )
    })

greedy_result = evaluate_prefix_quantities(
    "Next-marginal greedy heuristic",
    greedy_quantities
)
greedy_decisions = pd.DataFrame(greedy_decision_log)

display(greedy_result["Positive Plan"].round(4))
display(greedy_decisions.tail(10).round(4))

,Area,Category,Bicycles Relocated,Truck Space Used,Expected Profit
0,1,Child,10,10.0,336.3584
1,1,Electric,3,4.5,86.9649
2,3,Child,3,3.0,264.3762
3,3,Adult,3,4.5,186.5712
4,3,Mountain,2,3.4,157.4075
5,4,Child,7,7.0,392.0342
6,4,Adult,2,3.0,86.1784
7,4,Racing,5,8.5,300.1706
8,4,Mountain,17,28.9,992.8732
9,6,Adult,1,1.5,81.2511


,Iteration,Area,Category,Position,Marginal Profit,Profit per Space,Capacity Remaining
46,47,1,Child,8,42.2463,42.2463,14.3
47,48,1,Child,9,28.6548,28.6548,13.3
48,49,1,Child,10,25.9057,25.9057,12.3
49,50,6,Mountain,3,30.7437,18.0845,10.6
50,51,1,Electric,3,23.1995,15.4663,9.1
51,52,4,Racing,3,25.6256,15.0739,7.4
52,53,4,Racing,4,53.9240,31.7200,5.7
53,54,4,Racing,5,86.1784,50.6932,4.0
54,55,4,Mountain,16,21.9437,12.9081,2.3
55,56,4,Mountain,17,28.4976,16.7633,0.6


### 6.3 Model Comparison

The optimality gap is calculated against the exact MILP objective. A smaller gap is better. Capacity utilization is reported separately because filling the truck is not itself the business objective.

In [41]:
# 6.3 EXACT MODEL VERSUS FEASIBLE BASELINES

exact_quantities = {
    (area, category): int(
        full_plan.loc[
            (full_plan["Area"] == area)
            & (full_plan["Category"] == category),
            "Bicycles Relocated"
        ].iloc[0]
    )
    for area in full_areas
    for category in full_categories
}

exact_result = evaluate_prefix_quantities(
    "Exact MILP (HiGHS)",
    exact_quantities
)

if abs(
    exact_result["Expected Profit"] - full_objective_value
) > VALIDATION_TOLERANCE:
    raise AssertionError("Exact-plan evaluation does not match the MILP objective.")

benchmark_results = [
    exact_result,
    average_result,
    greedy_result
]
benchmark_optimum = exact_result["Expected Profit"]

benchmark_comparison = pd.DataFrame([
    {
        "Method": result["Method"],
        "Expected Profit": result["Expected Profit"],
        "Profit Difference from Optimal": (
            benchmark_optimum - result["Expected Profit"]
        ),
        "Optimality Gap (%)": 100 * (
            benchmark_optimum - result["Expected Profit"]
        ) / benchmark_optimum,
        "Bicycles Relocated": result["Bicycles Relocated"],
        "Truck Capacity Used": result["Truck Capacity Used"],
        "Capacity Utilization (%)": 100
        * result["Truck Capacity Used"]
        / FULL_TRUCK_CAPACITY,
        "Feasible": result["Feasible"]
    }
    for result in benchmark_results
]).sort_values(
    by="Expected Profit",
    ascending=False
 ).reset_index(drop=True)

best_method_name = benchmark_comparison.iloc[0]["Method"]
best_result = next(
    result
    for result in benchmark_results
    if result["Method"] == best_method_name
)

if best_method_name != "Exact MILP (HiGHS)":
    raise AssertionError(
        "A heuristic exceeded the solver-proven optimum; review the benchmark code."
    )

display(
    benchmark_comparison.style.format({
        "Expected Profit": "{:,.4f}",
        "Profit Difference from Optimal": "{:,.4f}",
        "Optimality Gap (%)": "{:.2f}%",
        "Truck Capacity Used": "{:.1f}",
        "Capacity Utilization (%)": "{:.2f}%"
    })
)

print(f"Best method: {best_method_name}")
print(f"Best expected profit: {best_result['Expected Profit']:,.4f}")

,Method,Expected Profit,Profit Difference from Optimal,Optimality Gap (%),Bicycles Relocated,Truck Capacity Used,Capacity Utilization (%),Feasible
0,Exact MILP (HiGHS),"4,358.7834",0.0000,0.00%,79,80.0,100.00%,True
1,Average-profit-density heuristic,"4,237.1973",121.5861,2.79%,80,80.0,100.00%,True
2,Next-marginal greedy heuristic,"3,076.9189","1,281.8645",29.41%,56,79.4,99.25%,True


Best method: Exact MILP (HiGHS)
Best expected profit: 4,358.7834


### 6.4 Export the Winning Plan for Business Use

The source workbook in `data/` is treated as read-only. The winning method's positive relocation decisions are written to the `OptimalRelocationPlan` worksheet in `results/BicyclesRelocationData_Optimized.xlsx`. The output workbook contains every original worksheet plus the generated plan; rerunning this cell safely replaces only the output file.

In [ ]:
# 6.4 EXPORT THE BEST PLAN TO THE BUSINESS WORKBOOK

import os
import tempfile

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.table import Table, TableStyleInfo

OUTPUT_SHEET_NAME = "OptimalRelocationPlan"
output_path.parent.mkdir(parents=True, exist_ok=True)

if best_method_name != "Exact MILP (HiGHS)":
    raise AssertionError("The exported plan must be the solver-proven best method.")

business_plan = best_result["Positive Plan"].copy()
business_plan.insert(
    3,
    "Truck Space per Bicycle",
    business_plan["Category"].map(full_space)
)

business_workbook = load_workbook(data_path)
source_sheet_names = [
    sheet_name
    for sheet_name in business_workbook.sheetnames
    if sheet_name != OUTPUT_SHEET_NAME
]

if OUTPUT_SHEET_NAME in business_workbook.sheetnames:
    del business_workbook[OUTPUT_SHEET_NAME]

business_sheet = business_workbook.create_sheet(OUTPUT_SHEET_NAME, 0)
business_sheet.sheet_view.showGridLines = False
business_sheet.freeze_panes = "A12"
business_sheet.sheet_properties.tabColor = "2F75B5"

title_fill = PatternFill("solid", fgColor="1F4E78")
section_fill = PatternFill("solid", fgColor="D9EAF7")
header_fill = PatternFill("solid", fgColor="5B9BD5")
total_fill = PatternFill("solid", fgColor="E2F0D9")
white_bold_font = Font(color="FFFFFF", bold=True)
bold_font = Font(bold=True)
thin_gray_border = Border(
    bottom=Side(style="thin", color="B7B7B7")
)

business_sheet.merge_cells("A1:F1")
business_sheet["A1"] = "RentalBike - Optimal Bicycle Relocation Plan"
business_sheet["A1"].fill = title_fill
business_sheet["A1"].font = Font(color="FFFFFF", bold=True, size=14)
business_sheet["A1"].alignment = Alignment(horizontal="center")

solution_summary = [
    ("Best Method", best_method_name),
    ("Solution Status", str(full_termination).title()),
    ("Optimal Expected Profit", best_result["Expected Profit"]),
    ("Bicycles Relocated", best_result["Bicycles Relocated"]),
    ("Truck Capacity Used", best_result["Truck Capacity Used"]),
    ("Truck Capacity Remaining", best_result["Truck Capacity Remaining"])
]

for row_number, (label, value) in enumerate(solution_summary, start=3):
    business_sheet.cell(row=row_number, column=1, value=label).font = bold_font
    business_sheet.cell(row=row_number, column=1).fill = section_fill
    business_sheet.cell(row=row_number, column=2, value=value)

business_sheet["B5"].number_format = "#,##0.0000"
business_sheet["B7"].number_format = "0.0"
business_sheet["B8"].number_format = "0.0"

business_sheet.merge_cells("H1:K1")
business_sheet["H1"] = "Category Stock Reconciliation"
business_sheet["H1"].fill = title_fill
business_sheet["H1"].font = white_bold_font
business_sheet["H1"].alignment = Alignment(horizontal="center")

category_headers = ["Category", "Available", "Relocated", "Remaining at Source"]

for column_number, header in enumerate(category_headers, start=8):
    cell = business_sheet.cell(row=3, column=column_number, value=header)
    cell.fill = header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center")

for row_number, category in enumerate(full_categories, start=4):
    relocated = int(
        best_result["Plan"].loc[
            best_result["Plan"]["Category"] == category,
            "Bicycles Relocated"
        ].sum()
    )
    values = [
        category,
        full_surplus[category],
        relocated,
        full_surplus[category] - relocated
    ]

    for column_number, value in enumerate(values, start=8):
        business_sheet.cell(row=row_number, column=column_number, value=value)

category_table = Table(
    displayName="CategoryStockReconciliation",
    ref=f"H3:K{3 + len(full_categories)}"
)
category_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)
business_sheet.add_table(category_table)

business_sheet.merge_cells("H11:K11")
business_sheet["H11"] = "Destination Area Reconciliation"
business_sheet["H11"].fill = section_fill
business_sheet["H11"].font = bold_font

area_headers = [
    "Area",
    "Bicycles Relocated",
    "Truck Space Used",
    "Expected Profit"
]

for column_number, header in enumerate(area_headers, start=8):
    cell = business_sheet.cell(row=12, column=column_number, value=header)
    cell.fill = header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

business_area_summary = best_result["Plan"].groupby(
    "Area",
    as_index=False
).agg({
    "Bicycles Relocated": "sum",
    "Truck Space Used": "sum",
    "Expected Profit": "sum"
})

for row_offset, values in enumerate(
    business_area_summary.itertuples(index=False, name=None)
):
    row_number = 13 + row_offset

    for column_number, value in enumerate(values, start=8):
        business_sheet.cell(row=row_number, column=column_number, value=value)

    business_sheet.cell(row=row_number, column=10).number_format = "0.0"
    business_sheet.cell(row=row_number, column=11).number_format = "#,##0.0000"

area_table = Table(
    displayName="DestinationAreaReconciliation",
    ref=f"H12:K{12 + len(business_area_summary)}"
)
area_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)
business_sheet.add_table(area_table)

business_sheet.merge_cells("H21:J21")
business_sheet["H21"] = "Solution Validation"
business_sheet["H21"].fill = section_fill
business_sheet["H21"].font = bold_font

validation_headers = ["Check", "Detail", "Passed"]

for column_number, header in enumerate(validation_headers, start=8):
    cell = business_sheet.cell(row=22, column=column_number, value=header)
    cell.fill = header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

for row_offset, validation_row in enumerate(
    full_validation.itertuples(index=False)
):
    row_number = 23 + row_offset
    business_sheet.cell(row=row_number, column=8, value=validation_row.Check)
    business_sheet.cell(row=row_number, column=9, value=validation_row.Detail)
    business_sheet.cell(row=row_number, column=10, value=bool(validation_row.Passed))

validation_table = Table(
    displayName="SolutionValidationChecks",
    ref=f"H22:J{22 + len(full_validation)}"
)
validation_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)
business_sheet.add_table(validation_table)

business_sheet.merge_cells("A10:F10")
business_sheet["A10"] = "Actionable Relocation Decisions"
business_sheet["A10"].fill = section_fill
business_sheet["A10"].font = bold_font

plan_headers = list(business_plan.columns)

for column_number, header in enumerate(plan_headers, start=1):
    cell = business_sheet.cell(row=11, column=column_number, value=header)
    cell.fill = header_fill
    cell.font = white_bold_font
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

first_plan_row = 12

for row_offset, values in enumerate(
    business_plan.itertuples(index=False, name=None)
):
    row_number = first_plan_row + row_offset

    for column_number, value in enumerate(values, start=1):
        business_sheet.cell(row=row_number, column=column_number, value=value)

    business_sheet.cell(row=row_number, column=4).number_format = "0.0"
    business_sheet.cell(row=row_number, column=5).number_format = "0.0"
    business_sheet.cell(row=row_number, column=6).number_format = "#,##0.0000"

last_plan_row = first_plan_row + len(business_plan) - 1

action_table = Table(
    displayName="OptimalRelocationActions",
    ref=f"A11:F{last_plan_row}"
)
action_table.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)
business_sheet.add_table(action_table)

total_row = last_plan_row + 1
business_sheet.cell(row=total_row, column=1, value="TOTAL").font = bold_font
business_sheet.cell(
    row=total_row,
    column=3,
    value=best_result["Bicycles Relocated"]
).font = bold_font
business_sheet.cell(
    row=total_row,
    column=5,
    value=best_result["Truck Capacity Used"]
).font = bold_font
business_sheet.cell(
    row=total_row,
    column=6,
    value=best_result["Expected Profit"]
).font = bold_font

for column_number in range(1, 7):
    business_sheet.cell(row=total_row, column=column_number).fill = total_fill
    business_sheet.cell(row=total_row, column=column_number).border = thin_gray_border

business_sheet.cell(row=total_row, column=5).number_format = "0.0"
business_sheet.cell(row=total_row, column=6).number_format = "#,##0.0000"

note_row = total_row + 2
business_sheet.cell(row=note_row, column=1, value="Note").font = bold_font
business_sheet.cell(
    row=note_row,
    column=2,
    value=(
        "Only positive relocation decisions are listed. Expected-profit units "
        "are preserved exactly as supplied in the source workbook."
    )
)
business_sheet.merge_cells(
    start_row=note_row,
    start_column=2,
    end_row=note_row,
    end_column=6
)

column_widths = {
    "A": 29,
    "B": 25,
    "C": 21,
    "D": 25,
    "E": 20,
    "F": 19,
    "G": 3,
    "H": 32,
    "I": 50,
    "J": 18,
    "K": 22
}

for column_letter, width in column_widths.items():
    business_sheet.column_dimensions[column_letter].width = width

temporary_handle = tempfile.NamedTemporaryFile(
    prefix=f".{output_path.stem}_",
    suffix=".xlsx",
    dir=output_path.parent,
    delete=False
)
temporary_path = Path(temporary_handle.name)
temporary_handle.close()

try:
    # Validate a temporary workbook before atomically replacing the output.
    business_workbook.save(temporary_path)
    business_workbook.close()

    verification_workbook = load_workbook(
        temporary_path,
        read_only=True,
        data_only=True
    )

    try:
        if OUTPUT_SHEET_NAME not in verification_workbook.sheetnames:
            raise AssertionError("The generated output worksheet is missing.")

        verified_source_sheets = [
            sheet_name
            for sheet_name in verification_workbook.sheetnames
            if sheet_name != OUTPUT_SHEET_NAME
        ]

        if verified_source_sheets != source_sheet_names:
            raise AssertionError("A source-data worksheet changed during export.")

        verification_sheet = verification_workbook[OUTPUT_SHEET_NAME]
        exported_quantity = sum(
            verification_sheet.cell(row=row_number, column=3).value
            for row_number in range(first_plan_row, last_plan_row + 1)
        )
        exported_profit = sum(
            verification_sheet.cell(row=row_number, column=6).value
            for row_number in range(first_plan_row, last_plan_row + 1)
        )

        if verification_sheet["B3"].value != best_method_name:
            raise AssertionError(
                "Exported best-method label does not match the comparison."
            )

        if exported_quantity != best_result["Bicycles Relocated"]:
            raise AssertionError("Exported bicycle quantity does not reconcile.")

        if abs(
            exported_profit - best_result["Expected Profit"]
        ) > VALIDATION_TOLERANCE:
            raise AssertionError("Exported expected profit does not reconcile.")
    finally:
        verification_workbook.close()

    try:
        os.replace(temporary_path, output_path)
    except PermissionError as error:
        raise PermissionError(
            f"Close {output_path.name} in Excel and rerun the export cell."
        ) from error
finally:
    business_workbook.close()

    if temporary_path.exists():
        temporary_path.unlink()

print(
    f"Exported {len(business_plan)} actionable rows to "
    f"{output_path} / {OUTPUT_SHEET_NAME}."
)

## 7.0 AI TOOL USAGE DISCLOSURE

### AI-assisted work

AI assistance was used to expand the simplified formulation into the complete data-driven Pyomo model, draft the workbook-validation logic, implement the two baseline heuristics, structure the comparison and result tables, create the business-workbook export, and propose post-solve checks.

### Validation of AI-assisted code

The implementation validates workbook schemas and numeric domains before modeling. After optimization, it requires an optimal solver termination and programmatically checks every binary value, every prefix link, category availability, truck capacity, missing-profit tail selections, and an independently recomputed objective value. Each heuristic is passed through the same feasibility evaluator, and the generated Excel tab is reopened to reconcile its method label, relocated quantity, and expected profit with the winning result.
The validation of AI code was made first making the AI follow the simplified modeling in the section 4.0, then with unitary tests, using pytest which tests critical functions of the modeling.

### Reflection

AI assistance accelerated implementation, especially for edge cases involving unequal profit-series lengths. It did not replace model verification, judment, system design and the planning of the solution.